# Load base solutions and rollouts

This notebook scans every `problem_*` folder under the target base-solution directory, loads `problem.json` and `solutions.json` when present, and gives a quick rollout preview.

In [7]:
from pathlib import Path
import json

BASE_DIR = Path(r"C:\Users\Preet Lodaya\Thought_Anchors\thought-anchors\math_rollouts_steered_VM\deepseek-r1-distill-qwen-14b\temperature_0.6_top_p_0.95\correct_base_solution")

def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def get_rollout(item):
    return item.get("rollout") or item.get("full_cot") or item.get("prompt") or ""

records = []
for problem_dir in sorted(p for p in BASE_DIR.iterdir() if p.is_dir()):
    problem_path = problem_dir / "problem.json"
    if not problem_path.exists():
        continue

    solution_path = problem_dir / "solutions.json"
    problem = load_json(problem_path)
    solutions = load_json(solution_path) if solution_path.exists() else []

    records.append({
        "problem_id": problem_dir.name,
        "problem": problem,
        "solutions": solutions,
        "has_solutions": solution_path.exists(),
        "num_solutions": len(solutions),
        "rollouts": [get_rollout(sol) for sol in solutions],
    })

len(records), [r["problem_id"] for r in records]


(5,
 ['problem_1591',
  'problem_2050',
  'problem_2137',
  'problem_2189',
  'problem_330'])

In [8]:
summary_rows = []
total_rollouts = 0
total_correct = 0

for record in records:
    solutions = record["solutions"]
    num_rollouts = len(solutions)
    num_correct = sum(1 for sol in solutions if sol.get("is_correct") is True)
    accuracy = (num_correct / num_rollouts) if num_rollouts else None

    total_rollouts += num_rollouts
    total_correct += num_correct
    summary_rows.append({
        "problem_id": record["problem_id"],
        "num_rollouts": num_rollouts,
        "num_correct": num_correct,
        "accuracy": accuracy,
    })

overall_accuracy = (total_correct / total_rollouts) if total_rollouts else None

print(f"overall_rollouts={total_rollouts}")
print(f"overall_correct={total_correct}")
print(f"overall_accuracy={overall_accuracy:.3f}" if overall_accuracy is not None else "overall_accuracy=<none>")
print()

for row in summary_rows:
    acc = f"{row['accuracy']:.3f}" if row["accuracy"] is not None else "<none>"
    print(f"{row['problem_id']}: rollouts={row['num_rollouts']} correct={row['num_correct']} accuracy={acc}")


overall_rollouts=155
overall_correct=121
overall_accuracy=0.781

problem_1591: rollouts=30 correct=23 accuracy=0.767
problem_2050: rollouts=25 correct=23 accuracy=0.920
problem_2137: rollouts=25 correct=20 accuracy=0.800
problem_2189: rollouts=25 correct=20 accuracy=0.800
problem_330: rollouts=50 correct=35 accuracy=0.700


In [9]:
for record in records:
    problem = record["problem"]
    print(f"{record['problem_id']}: {problem.get('type')} | {problem.get('level')} | gt={problem.get('gt_answer')}")
    print(f"  has_solutions={record['has_solutions']} | num_solutions={record['num_solutions']}")

    if record["rollouts"]:
        preview = record["rollouts"][0][:350].replace("\n", " ")
        print(f"  rollout_preview={preview}")
    else:
        print("  rollout_preview=<none>")
    print()


problem_1591: Algebra | Level 5 | gt=6.17
  has_solutions=True | num_solutions=30
  rollout_preview=Okay, so I've got this math problem here about bank accounts and interest rates. Let me try to figure it out step by step.   The problem says that Dr. Fu Manchu has a bank account with an annual interest rate of 6 percent, but it compounds monthly. We need to find the equivalent annual interest rate, r percent, that would give the same return if it

problem_2050: Counting & Probability | Level 5 | gt=336
  has_solutions=True | num_solutions=25
  rollout_preview=Okay, so I'm trying to solve this problem where I have an unlimited supply of congruent equilateral triangles, each colored one of six different colors. I need to construct a large equilateral triangle using four of these small triangles. The key here is that two large triangles are considered the same (indistinguishable) if one can be transformed 

problem_2137: Counting & Probability | Level 5 | gt=640
  has_solutions=True | num

## Base vs steered comparison

This section compares rollout accuracy between the base test set and the steered test set.

In [10]:
BASE_TEST_DIR = Path(r"C:\Users\Preet Lodaya\Thought_Anchors\thought-anchors\math_rollouts\deepseek-r1-distill-qwen-14b\temperature_0.6_top_p_0.95\correct_base_solution\test")
STEERED_TEST_DIR = Path(r"C:\Users\Preet Lodaya\Thought_Anchors\thought-anchors\math_rollouts_steered_VM\deepseek-r1-distill-qwen-14b\temperature_0.6_top_p_0.95\correct_base_solution\test")

def load_problem_rollout_summary(root_dir: Path):
    rows = {}
    for problem_dir in sorted(p for p in root_dir.iterdir() if p.is_dir()):
        problem_path = problem_dir / "problem.json"
        solutions_path = problem_dir / "solutions.json"
        if not problem_path.exists() or not solutions_path.exists():
            continue

        problem = load_json(problem_path)
        solutions = load_json(solutions_path)
        num_rollouts = len(solutions)
        num_correct = sum(1 for sol in solutions if sol.get("is_correct") is True)
        rows[problem_dir.name] = {
            "problem_id": problem_dir.name,
            "problem": problem,
            "num_rollouts": num_rollouts,
            "num_correct": num_correct,
            "accuracy": (num_correct / num_rollouts) if num_rollouts else None,
        }
    return rows

base_rows = load_problem_rollout_summary(BASE_TEST_DIR)
steered_rows = load_problem_rollout_summary(STEERED_TEST_DIR)
shared_problem_ids = sorted(set(base_rows) & set(steered_rows))
missing_in_steered = sorted(set(base_rows) - set(steered_rows))
missing_in_base = sorted(set(steered_rows) - set(base_rows))

print(f"base_problems={len(base_rows)}")
print(f"steered_problems={len(steered_rows)}")
print(f"shared_problems={len(shared_problem_ids)}")
print(f"missing_in_steered={missing_in_steered}")
print(f"missing_in_base={missing_in_base}")
print()

for problem_id in shared_problem_ids:
    base = base_rows[problem_id]
    steered = steered_rows[problem_id]
    base_acc = base["accuracy"]
    steered_acc = steered["accuracy"]
    delta = None if base_acc is None or steered_acc is None else steered_acc - base_acc
    delta_text = f"{delta:+.3f}" if delta is not None else "<none>"
    base_text = f"{base_acc:.3f}" if base_acc is not None else "<none>"
    steered_text = f"{steered_acc:.3f}" if steered_acc is not None else "<none>"
    print(
        f"{problem_id}: base={base_text} ({base['num_correct']}/{base['num_rollouts']}) | "
        f"steered={steered_text} ({steered['num_correct']}/{steered['num_rollouts']}) | delta={delta_text}"
    )


base_problems=4
steered_problems=4
shared_problems=4
missing_in_steered=[]
missing_in_base=[]

problem_1923: base=1.000 (30/30) | steered=1.000 (30/30) | delta=+0.000
problem_2542: base=1.000 (30/30) | steered=1.000 (30/30) | delta=+0.000
problem_3009: base=0.800 (24/30) | steered=0.833 (20/24) | delta=+0.033
problem_489: base=1.000 (30/30) | steered=1.000 (30/30) | delta=+0.000


## Chunk and token distributions

This section summarizes `usage.completion_tokens` and `chunk_context.num_chunks_generated` for both base and steered rollouts.

In [11]:
import math

def extract_metrics(root_dir: Path):
    rows = []
    for problem_dir in sorted(p for p in root_dir.iterdir() if p.is_dir()):
        solutions_path = problem_dir / "solutions.json"
        if not solutions_path.exists():
            continue
        solutions = load_json(solutions_path)
        for idx, sol in enumerate(solutions):
            usage = sol.get("usage") or {}
            chunk_context = sol.get("chunk_context") or {}
            rows.append({
                "problem_id": problem_dir.name,
                "solution_idx": idx,
                "completion_tokens": usage.get("completion_tokens"),
                "total_tokens": usage.get("total_tokens"),
                "num_chunks_generated": chunk_context.get("num_chunks_generated"),
                "is_correct": sol.get("is_correct") is True,
            })
    return rows

def summarize(values):
    values = [v for v in values if v is not None]
    if not values:
        return {"count": 0}
    values = sorted(values)
    n = len(values)
    mean = sum(values) / n
    variance = sum((x - mean) ** 2 for x in values) / n
    def q(p):
        if n == 1:
            return values[0]
        pos = (n - 1) * p
        lo = math.floor(pos)
        hi = math.ceil(pos)
        if lo == hi:
            return values[int(pos)]
        return values[lo] + (values[hi] - values[lo]) * (pos - lo)
    return {
        "count": n,
        "mean": mean,
        "std": math.sqrt(variance),
        "min": values[0],
        "p25": q(0.25),
        "median": q(0.5),
        "p75": q(0.75),
        "max": values[-1],
    }

def print_summary(label, rows):
    completion = summarize(r["completion_tokens"] for r in rows)
    chunks = summarize(r["num_chunks_generated"] for r in rows)
    correct_rows = [r for r in rows if r["is_correct"]]
    incorrect_rows = [r for r in rows if not r["is_correct"]]
    print()
    print(f"{label} | rows={len(rows)} | correct={len(correct_rows)} | incorrect={len(incorrect_rows)}")
    print(f"  completion_tokens: {completion}")
    print(f"  num_chunks_generated: {chunks}")
    print()

base_metric_rows = extract_metrics(BASE_TEST_DIR)
steered_metric_rows = extract_metrics(STEERED_TEST_DIR)

print_summary("base", base_metric_rows)
print_summary("steered", steered_metric_rows)

all_problem_ids = sorted(set(r["problem_id"] for r in base_metric_rows) | set(r["problem_id"] for r in steered_metric_rows))
for problem_id in all_problem_ids:
    base_rows = [r for r in base_metric_rows if r["problem_id"] == problem_id]
    steered_rows = [r for r in steered_metric_rows if r["problem_id"] == problem_id]
    print(problem_id)
    if base_rows:
        print_summary(f"base::{problem_id}", base_rows)
    if steered_rows:
        print_summary(f"steered::{problem_id}", steered_rows)



base | rows=120 | correct=114 | incorrect=6
  completion_tokens: {'count': 120, 'mean': 3296.425, 'std': 2945.790781387628, 'min': 723, 'p25': 1104.5, 'median': 1600.5, 'p75': 4687.0, 'max': 11337}
  num_chunks_generated: {'count': 120, 'mean': 140.55833333333334, 'std': 104.574598240788, 'min': 35, 'p25': 55.0, 'median': 84.5, 'p75': 210.0, 'max': 400}


steered | rows=114 | correct=110 | incorrect=4
  completion_tokens: {'count': 114, 'mean': 2992.0701754385964, 'std': 2920.1991151749207, 'min': 683, 'p25': 1045.0, 'median': 1540.0, 'p75': 3689.25, 'max': 12207}
  num_chunks_generated: {'count': 114, 'mean': 129.21929824561403, 'std': 102.09273036063831, 'min': 31, 'p25': 53.25, 'median': 83.0, 'p75': 178.0, 'max': 418}

problem_1923

base::problem_1923 | rows=30 | correct=30 | incorrect=0
  completion_tokens: {'count': 30, 'mean': 3327.733333333333, 'std': 1341.0452374505824, 'min': 1391, 'p25': 2265.25, 'median': 2900.0, 'p75': 4143.25, 'max': 7216}
  num_chunks_generated: {'count

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

RESAMPLED_BASE_DIR = Path(r"C:\Users\Preet Lodaya\Thought_Anchors\thought-anchors\math_rollouts\deepseek-r1-distill-qwen-14b\temperature_0.6_top_p_0.95\correct_base_solution")
SIMILARITY_MODEL_NAME = "all-MiniLM-L6-v2"
_similarity_model_cache = {}

def get_similarity_model(model_name: str = SIMILARITY_MODEL_NAME):
    if model_name not in _similarity_model_cache:
        _similarity_model_cache[model_name] = SentenceTransformer(model_name)
    return _similarity_model_cache[model_name]

def cosine_similarity(vec_a, vec_b):
    denom = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    return float(np.dot(vec_a, vec_b) / denom) if denom else None

def get_resampled_chunk_similarities(problem_id, chunk_idx, base_dir=RESAMPLED_BASE_DIR, model_name: str = SIMILARITY_MODEL_NAME):
    problem_name = str(problem_id)
    if not problem_name.startswith("problem_"):
        problem_name = f"problem_{problem_name}"

    solutions_path = Path(base_dir) / problem_name / f"chunk_{chunk_idx}" / "solutions.json"
    if not solutions_path.exists():
        raise FileNotFoundError(f"No resampled solutions found at {solutions_path}")

    solutions = load_json(solutions_path)
    model = get_similarity_model(model_name)
    rows = []
    texts_to_embed = []
    pending_pairs = []

    for solution_idx, sol in enumerate(solutions):
        removed = sol.get("chunk_removed")
        resampled = sol.get("chunk_resampled")

        row = {
            "solution_idx": solution_idx,
            "chunk_removed": removed,
            "chunk_resampled": resampled,
            "is_correct": sol.get("is_correct"),
            "answer": sol.get("answer"),
            "similarity": None,
        }
        rows.append(row)

        if isinstance(removed, str) and isinstance(resampled, str):
            pending_pairs.append(solution_idx)
            texts_to_embed.extend([removed, resampled])

    if pending_pairs:
        embeddings = model.encode(texts_to_embed, convert_to_numpy=True)
        for pair_idx, solution_idx in enumerate(pending_pairs):
            removed_embedding = embeddings[2 * pair_idx]
            resampled_embedding = embeddings[2 * pair_idx + 1]
            rows[solution_idx]["similarity"] = cosine_similarity(removed_embedding, resampled_embedding)

    valid_similarities = [row["similarity"] for row in rows if row["similarity"] is not None]
    return {
        "problem_id": problem_name,
        "chunk_idx": int(chunk_idx),
        "solutions_path": str(solutions_path),
        "num_resamples": len(rows),
        "num_valid_pairs": len(valid_similarities),
        "mean_similarity": float(np.mean(valid_similarities)) if valid_similarities else None,
        "min_similarity": float(np.min(valid_similarities)) if valid_similarities else None,
        "max_similarity": float(np.max(valid_similarities)) if valid_similarities else None,
        "rows": rows,
    }

example = get_resampled_chunk_similarities("problem_1591", 0)
{k: v for k, v in example.items() if k != "rows"}
